# Multi-year urban green statistics — demonstrator

SATDES Task 1, Subtask 1.6. This notebook is a thin front end over the
`urban_green` package (see `../src/urban_green/`): it does not
reimplement any logic, it just calls the same pipeline the CLI
(`python -m urban_green.cli`) runs, and displays the results inline.

Set `DES_USERNAME` / `DES_PASSWORD` in the environment before running
this notebook for real Digital Earth Sweden access; otherwise it falls
back to the placeholder test credentials.

In [ ]:
import geopandas as gpd
import pandas as pd
from IPython.display import Image, display

from urban_green.config import PipelineConfig
from urban_green.pipeline import run

## 1. Configure the run

Point `ROI_PATH` at any vector file covering part of Sweden (Shapefile,
GeoPackage, GeoParquet, GeoJSON, ...).

In [ ]:
ROI_PATH = "path/to/region_of_interest.gpkg"
YEARS = list(range(2018, 2026))
OUTPUT_DIR = "./outputs_demo"
REGION_ID_FIELD = None  # set to a column name for per-sub-region statistics

config = PipelineConfig(
    roi_path=ROI_PATH,
    years=YEARS,
    output_dir=OUTPUT_DIR,
    region_id_field=REGION_ID_FIELD,
)

## 2. Preview the ROI

In [ ]:
roi_preview = gpd.read_parquet(ROI_PATH) if str(ROI_PATH).endswith(".parquet") else gpd.read_file(ROI_PATH)
roi_preview.explore()

## 3. Run the pipeline

This acquires annual max-NDVI from Digital Earth Sweden for every tile x
year, stitches it, thresholds it, computes statistics and the change
product, and writes figures — the same steps the CLI runs.

In [ ]:
outputs_dir = run(config)
outputs_dir

## 4. Inspect results

In [ ]:
stats = pd.read_csv(outputs_dir / "green_area_statistics.csv")
stats

In [ ]:
change_stats = pd.read_csv(outputs_dir / "change_statistics.csv")
change_stats

In [ ]:
figures_dir = outputs_dir.parent / "figures"
display(Image(filename=str(figures_dir / "ndvi_distribution.png")))
display(Image(filename=str(figures_dir / "green_area_timeseries.png")))

## 5. (Optional) per-region breakdown

Only produced when `REGION_ID_FIELD` was set above.

In [ ]:
by_region_path = outputs_dir / "green_area_statistics_by_region.csv"
if by_region_path.exists():
    display(pd.read_csv(by_region_path))
else:
    print("No region_id_field was set for this run.")